<a href="https://colab.research.google.com/github/Svein-Tore/colab/blob/main/demo_SIMC_til_heftet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [70]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
y0_input = widgets.FloatText(value=20.0, description='y0 (Start):')
ysp_input = widgets.FloatText(value=65.0, description='y_sp (Mål):')
K_slider = widgets.FloatSlider(value=13.01, min=1.0, max=30.0, step=0.01, description='K (Prosessf.):')
T_slider = widgets.IntSlider(value=130, min=10, max=600, step=10, description='T (Tidsk.):')
L_slider = widgets.IntSlider(value=4, min=0, max=50, step=1, description='L (Dødtid):')
sprang_slider = widgets.IntSlider(value=38, min=0, max=50, step=1, description='start sprang:')
lambda_dropdown = widgets.Dropdown(options=(1, 2, 4, 6, 8, 10), value=1, description='\u03BB-faktor:')
metning_checkbox = widgets.Checkbox(value=False, description='Aktiver pådragsmetning (0-100%)')
kontroll_panel = widgets.VBox([widgets.Label(value="NIVÅINNSTILLINGER"), y0_input, ysp_input, sprang_slider, widgets.HTML(value="<br>"), widgets.Label(value="PROSESSENS EGENSKAPER"), K_slider, T_slider, L_slider, widgets.HTML(value="<br>"), widgets.Label(value="REGULATOR HASTIGHET / MODUS"), lambda_dropdown, metning_checkbox], layout=widgets.Layout(margin='40px 0px 0px 30px'))
plot_utdata = widgets.Output()
def simuler_og_plott(change=None):
    y0 = y0_input.value
    y_sp = ysp_input.value
    K = K_slider.value
    T = T_slider.value
    L = L_slider.value
    sprang_tid = sprang_slider.value
    lambda_faktor = lambda_dropdown.value
    bruk_metning = metning_checkbox.value
    u0 = y0 / K
    lambda_val = max(L, T / lambda_faktor)
    Kp = (1 / K) * (T / (lambda_val + L))
    Ti = min(T, 4 * (lambda_val + L))
    dt = 0.5
    t_slutt = 5 * T
    t_vektor = np.arange(0, t_slutt, dt)
    N = len(t_vektor)
    settpunkt = np.ones(N) * y0
    y_aapen = np.ones(N) * y0
    y_lukket = np.ones(N) * y0
    u_lukket = np.ones(N) * u0
    u_aapen_sprangverdi = u0 + (y_sp - y0) / K
    dødtid_steps = int(round(L / dt))
    u_historikk_lukket = [u0] * (dødtid_steps + 1)
    u_historikk_aapen = [u0] * (dødtid_steps + 1)
    integratør = 0.0
    for k in range(0, N - 1):
        if t_vektor[k] >= sprang_tid:
            settpunkt[k+1] = y_sp
            u_aapen_naa = u_aapen_sprangverdi
        else:
            settpunkt[k+1] = y0
            u_aapen_naa = u0
        avvik = settpunkt[k] - y_lukket[k]
        P_ledd = Kp * avvik
        integratør += (Kp / Ti) * avvik * dt
        u_temp = u0 + P_ledd + integratør
        if bruk_metning:
            u_lukket[k] = np.clip(u_temp, 0.0, 100.0)
        else:
            u_lukket[k] = u_temp
        u_historikk_lukket.append(u_lukket[k])
        u_forsinket_lukket = u_historikk_lukket[- (dødtid_steps + 1)]
        derivert_y_lukket = (-(y_lukket[k] - y0) + K * (u_forsinket_lukket - u0)) / T
        y_lukket[k+1] = y_lukket[k] + derivert_y_lukket * dt
        u_historikk_aapen.append(u_aapen_naa)
        u_forsinket_aapen = u_historikk_aapen[- (dødtid_steps + 1)]
        derivert_y_aapen = (-(y_aapen[k] - y0) + K * (u_forsinket_aapen - u0)) / T
        y_aapen[k+1] = y_aapen[k] + derivert_y_aapen * dt
    u_lukket[N-1] = u_lukket[N-2]
    sprang_amplitude = y_sp - y0
    abs_amplitude = abs(sprang_amplitude)
    y_632 = y0 + 0.632 * sprang_amplitude
    y_982 = y0 + 0.982 * sprang_amplitude
    t_lambda_linje = None
    t_T_linje = None
    t_4lambda_linje = None
    t_4T_linje = None
    if not bruk_metning:
        v_lambda = lambda_val
        v_T = float(T)
        v_4lambda = 4 * lambda_val
        v_4T = 4.0 * T
        t_lambda_linje = sprang_tid + L + v_lambda
        t_T_linje = sprang_tid + L + v_T
        t_4lambda_linje = sprang_tid + L + v_4lambda
        t_4T_linje = sprang_tid + L + v_4T
    else:
        for idx in range(N):
            tid = t_vektor[idx]
            if tid >= sprang_tid + L:
                if t_lambda_linje is None:
                    if (sprang_amplitude >= 0 and y_lukket[idx] >= y_632) or (sprang_amplitude < 0 and y_lukket[idx] <= y_632): t_lambda_linje = tid
                if t_T_linje is None:
                    if (sprang_amplitude >= 0 and y_aapen[idx] >= y_632) or (sprang_amplitude < 0 and y_aapen[idx] <= y_632): t_T_linje = tid
                if t_4lambda_linje is None:
                    if (sprang_amplitude >= 0 and y_lukket[idx] >= y_982) or (sprang_amplitude < 0 and y_lukket[idx] <= y_982): t_4lambda_linje = tid
                if t_4T_linje is None:
                    if (sprang_amplitude >= 0 and y_aapen[idx] >= y_982) or (sprang_amplitude < 0 and y_aapen[idx] <= y_982): t_4T_linje = tid
        v_lambda = t_lambda_linje - sprang_tid - L if t_lambda_linje else 0
        v_T = t_T_linje - sprang_tid - L if t_T_linje else 0
        v_4lambda = t_4lambda_linje - sprang_tid - L if t_4lambda_linje else 0
        v_4T = t_4T_linje - sprang_tid - L if t_4T_linje else 0
    with plot_utdata:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(9.5, 8.5))
        tekst = "stasjonærverdi" if lambda_faktor == 1 else "set-punkt"
        ax1.axhline(y=y_sp, color='red', linestyle='--', linewidth=2.5, label=str(tekst) + " (" + str(round(y_sp,1)) + "%)")
        if lambda_faktor > 1: ax1.plot(t_vektor, y_lukket, 'b-', label='Lambda-regulering', linewidth=3.5)
        ax1.plot(t_vektor, y_aapen, 'g-', label='Åpen sløyfe', linewidth=3.5)
        ax1.axhline(y=y_632, color='#555555', linestyle='--', linewidth=1.5)
        ax1.axhline(y=y_982, color='#888888', linestyle='-.', linewidth=1.5)
        stasjonær_verdi = y0 + sprang_amplitude
        høyeste_kurvepunkt = max(y0, y_sp, stasjonær_verdi)
        laveste_kurvepunkt = min(y0, y_sp, stasjonær_verdi)
        if lambda_faktor > 1:
            topp_verdi = høyeste_kurvepunkt + max(abs_amplitude * 0.25, 12)
            bunn_verdi = laveste_kurvepunkt - max(abs_amplitude * 0.25, 12)
        else:
            topp_verdi = høyeste_kurvepunkt + max(abs_amplitude * 0.15, 6)
            bunn_verdi = laveste_kurvepunkt - max(abs_amplitude * 0.15, 6)
        if bunn_verdi < 0: bunn_verdi = 0
        y_retning = 1 if sprang_amplitude >= 0 else -1
        offset_632 = abs_amplitude * 0.02 if sprang_amplitude >= 0 else -abs_amplitude * 0.06
        offset_982 = abs_amplitude * 0.02 if sprang_amplitude >= 0 else -abs_amplitude * 0.06
        x_prosent_tekst = t_slutt * 0.8
        ax1.text(x_prosent_tekst, y_982 + offset_982, "98.2% (" + str(round(y_982,1)) + "%)", color='#555555', fontsize=11, fontweight='bold', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
        ax1.text(x_prosent_tekst, y_632 + offset_632, "63.2% (" + str(round(y_632,1)) + "%)", color='#333333', fontsize=11, fontweight='bold', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
        ax1.axvline(x=sprang_tid, color='red', linestyle=':', linewidth=1.5)
        ax1.text(sprang_tid + 3, topp_verdi - (topp_verdi-bunn_verdi)*0.08, 'Sprang t=' + str(sprang_tid) + 's', color='red', fontsize=10, fontweight='bold')
        if lambda_faktor > 1 and t_lambda_linje is not None and t_lambda_linje < t_slutt:
            ax1.axvline(x=t_lambda_linje, color='blue', linestyle=':', linewidth=2)
            y_pos_lam = y_632 - (abs_amplitude * 0.12 * y_retning)
            ax1.text(t_lambda_linje - 2, y_pos_lam, '\u03BB = ' + str(round(v_lambda,1)) + 's\n(t = ' + str(round(t_lambda_linje,1)) + 's)', color='blue', fontsize=10, fontweight='bold', horizontalalignment='right')
        if t_T_linje is not None and t_T_linje < t_slutt:
            ax1.axvline(x=t_T_linje, color='green', linestyle=':', linewidth=2)
            y_pos_T = y_632 + (abs_amplitude * 0.05 * y_retning)
            ax1.text(t_T_linje + 3, y_pos_T, 'T = ' + str(round(v_T,0)) + 's\n(t = ' + str(round(t_T_linje,1)) + 's)', color='green', fontsize=10, fontweight='bold')
        if lambda_faktor > 1 and t_4lambda_linje is not None and t_4lambda_linje < t_slutt:
            ax1.axvline(x=t_4lambda_linje, color='#4ba3e3', linestyle='--', linewidth=1.5)
            y_pos_4lam = y_982 - (abs_amplitude * 0.12 * y_retning)
            ax1.text(t_4lambda_linje - 2, y_pos_4lam, '4\u03BB = ' + str(round(v_4lambda,1)) + 's\n(t = ' + str(round(t_4lambda_linje,1)) + 's)', color='#1d6fa5', fontsize=10, fontweight='bold', horizontalalignment='right')
        if t_4T_linje is not None and t_4T_linje < t_slutt:
            ax1.axvline(x=t_4T_linje, color='#5cb85c', linestyle='--', linewidth=1.5)
            y_pos_4T = y_982 + (abs_amplitude * 0.1 * y_retning)
            ax1.text(t_4T_linje + 3, y_pos_4T, '4T = ' + str(round(v_4T,0)) + 's\n(t = ' + str(round(t_4T_linje,1)) + 's)', color='darkgreen', fontsize=10, fontweight='bold')
        t_dødtid = sprang_tid + L
        if L > 0 and t_dødtid < t_slutt:
            ax1.axvline(x=t_dødtid, color='purple', linestyle=':', linewidth=1.5)
            ax1.text(t_dødtid - 2, y0 + (abs_amplitude * 0.05 * y_retning), 'L = ' + str(L) + 's', color='purple', fontsize=10, fontweight='bold', horizontalalignment='right')
        ax1.set_ylim(bunn_verdi, topp_verdi)
        ax2.set_ylim(0, 105)
        ax1.set_ylabel('Nivå [%]', fontsize=12)
        ax1.legend(loc='lower right')
        ax1.grid(True)
        if lambda_faktor > 1: ax2.plot(t_vektor, u_lukket, 'b-', label='U_lukket', linewidth=2.5)
        ax2.plot(t_vektor, np.ones(N) * u_aapen_sprangverdi, 'g--', label='U_open', linewidth=2.5)
        ax2.axvline(x=sprang_tid, color='red', linestyle=':', linewidth=1.5)
        ax2.set_xlabel('Tid [sekunder]', fontsize=12)
        ax2.set_ylabel('Pådrag [%]', fontsize=12)
        ax2.legend(loc='lower right')
        ax2.grid(True)
        plt.tight_layout()
        plt.show()
y0_input.observe(simuler_og_plott, names='value')
ysp_input.observe(simuler_og_plott, names='value')
K_slider.observe(simuler_og_plott, names='value')
T_slider.observe(simuler_og_plott, names='value')
L_slider.observe(simuler_og_plott, names='value')
sprang_slider.observe(simuler_og_plott, names='value')
lambda_dropdown.observe(simuler_og_plott, names='value')
metning_checkbox.observe(simuler_og_plott, names='value')
simuler_og_plott()
display(widgets.HBox([kontroll_panel, plot_utdata]))